# Check `dev` stack

Verifies all developer tools are properly installed and functional.
Each section performs a version check followed by a self-contained functional test.

## Version checks

In [ ]:
! quarto --version

In [ ]:
! decktape --version

In [ ]:
! tippecanoe --version

In [ ]:
! jekyll --version

In [ ]:
! gpq --version

## Functional tests

### `quarto` — HTML render

Renders a minimal document to HTML. No network access required.

In [ ]:
import os

qmd = """\
---
title: "GDS Test"
format: html
---

# Test

Quarto functional test.
"""

with open('/tmp/gds_test.qmd', 'w') as f:
    f.write(qmd)

In [ ]:
! quarto render /tmp/gds_test.qmd 2>&1

In [ ]:
assert os.path.exists('/tmp/gds_test.html'), 'quarto render failed: no HTML output'
html_size = os.path.getsize('/tmp/gds_test.html')
assert html_size > 1000, f'quarto render failed: HTML suspiciously small ({html_size} bytes)'
print(f'PASS: quarto rendered HTML ({html_size:,} bytes)')
os.remove('/tmp/gds_test.html')
os.remove('/tmp/gds_test.qmd')

### `decktape` — PDF from reveal.js presentation

Generates a self-contained reveal.js presentation with quarto (no CDN required since
quarto bundles reveal.js locally), then converts it to PDF with decktape.
Uses `--disable-dev-shm-usage` to prevent Chrome crashes in Docker containers
where /dev/shm is limited.

In [ ]:
slides_qmd = """\
---
title: "GDS Slides Test"
format:
  revealjs:
    embed-resources: true
---

## Slide 1

Testing decktape integration.

## Slide 2

Decktape is properly installed.
"""

with open('/tmp/gds_slides.qmd', 'w') as f:
    f.write(slides_qmd)

In [ ]:
! quarto render /tmp/gds_slides.qmd 2>&1

In [ ]:
import subprocess

result = subprocess.run(
    [
        'decktape', 'reveal',
        '--chrome-arg=--no-sandbox',
        '--chrome-arg=--disable-dev-shm-usage',
        '-s', '1280x960',
        'file:///tmp/gds_slides.html',
        '/tmp/gds_slides.pdf',
    ],
    capture_output=True,
    text=True,
)
print(result.stdout)
if result.returncode != 0:
    print('--- stderr ---')
    print(result.stderr)
    raise RuntimeError(f'decktape exited with code {result.returncode}')

In [ ]:
assert os.path.exists('/tmp/gds_slides.pdf'), 'decktape failed: PDF not created'
pdf_size = os.path.getsize('/tmp/gds_slides.pdf')
assert pdf_size > 1000, f'decktape failed: PDF suspiciously small ({pdf_size} bytes)'
print(f'PASS: decktape created PDF ({pdf_size:,} bytes)')
for path in ['/tmp/gds_slides.qmd', '/tmp/gds_slides.html', '/tmp/gds_slides.pdf']:
    if os.path.exists(path):
        os.remove(path)

### `tippecanoe` — GeoJSON to MBTiles

Converts an inline GeoJSON (no download required) to a vector tile archive.

In [ ]:
import json

geojson = {
    'type': 'FeatureCollection',
    'features': [
        {
            'type': 'Feature',
            'geometry': {'type': 'Point', 'coordinates': [-0.1278, 51.5074]},
            'properties': {'name': 'London'},
        },
        {
            'type': 'Feature',
            'geometry': {'type': 'Point', 'coordinates': [2.3522, 48.8566]},
            'properties': {'name': 'Paris'},
        },
    ],
}

with open('/tmp/test_points.geojson', 'w') as f:
    json.dump(geojson, f)

In [ ]:
! tippecanoe -o /tmp/test.mbtiles /tmp/test_points.geojson 2>&1

In [ ]:
assert os.path.exists('/tmp/test.mbtiles'), 'tippecanoe failed: .mbtiles not created'
mbtiles_size = os.path.getsize('/tmp/test.mbtiles')
assert mbtiles_size > 0, 'tippecanoe failed: .mbtiles is empty'
print(f'PASS: tippecanoe created MBTiles ({mbtiles_size:,} bytes)')
os.remove('/tmp/test.mbtiles')
os.remove('/tmp/test_points.geojson')